# 00 — `min_delta` calibration for early stopping (task T6)

Short run of the query-embedding model (reference seed, dropout = first grid value, `calibration_epochs` epochs, no early stopping). `min_delta` = `min_delta_se_multiple` × the user-clustered standard error of the per-row change in **autoregressive valid NDCG@10** between the last two epochs (`nbp.hybrid.fusion.cluster_se`): a gain smaller than that is noise. Writes `results/min_delta_calibration.json` and fills `min_delta` in `configs/train.yaml`. Plan: `docs/plans/plan_oct_12_new_split_retrain.md` (section 5).

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # torch before pandas-importing modules (Windows c10.dll)
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from nbp.data.split import TRAIN
from nbp.eval import runner as R
from nbp.eval.report import metrics, row_set_masks, write_json
from nbp.paths import CONFIGS, EXPERIMENTS, INTERIM, PROCESSED

SPLIT = yaml.safe_load((CONFIGS / "split.yaml").read_text(encoding="utf-8"))
TRAIN_CFG = yaml.safe_load((CONFIGS / "train.yaml").read_text(encoding="utf-8"))
MODEL_CFG = yaml.safe_load((CONFIGS / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
assert MODEL_CFG["MAX_ITEM_LIST_LENGTH"] == SPLIT["max_history"], "history length differs between configs"
SEEDS = list(SPLIT["seeds"])
REF_SEED = SEEDS[0]  # grids and sameDest weights are chosen on this seed
FAST = os.environ.get("NBP_FAST") == "1"  # smoke run: a user subsample, short training, outputs under *_fast
FAST_FRAC = float(os.environ.get("NBP_FAST_FRAC", "0.02"))
if FAST:
    TRAIN_CFG["max_epochs"] = int(os.environ.get("NBP_FAST_EPOCHS", "2"))
RES = ROOT / ("results_fast" if FAST else "results")
RANKS = INTERIM / ("week6_fast" if FAST else "week6")
EXP = EXPERIMENTS / "_fast" if FAST else EXPERIMENTS
TODAY = f"{date.today():%Y-%m-%d}"
K_MON = int(TRAIN_CFG["monitor"].split("@")[1])
BATCH = int(TRAIN_CFG["eval_batch"])
torch.set_num_threads(int(os.environ.get("NBP_THREADS", torch.get_num_threads())))

rows_full = R.load_rows(PROCESSED / "expedia_date_split" / "rows.npz")
codes = rows_full["codes"]  # (N, F) query codes, row = global event index
N_ITEMS = int(rows_full["y"].max()) + 1  # model ids 0 (padding) .. number of clusters
rows = R.subsample_users(rows_full, FAST_FRAC, 0) if FAST else rows_full
views = R.window_views(rows)
idx = R.window_index(rows)
Y = {w: rows["y"][i].astype(np.int64) for w, i in idx.items()}
L = {w: rows["L"][i] for w, i in idx.items()}
H = {w: rows["h"][i] for w, i in idx.items()}
USERS = {w: rows["user"][i] for w, i in idx.items()}
probe, y_probe = R.train_probe_view(rows, TRAIN_CFG["train_probe_rows"], TRAIN_CFG["train_probe_seed"])
print("FAST" if FAST else "FULL", {w: len(v) for w, v in views.items()}, "| items", N_ITEMS, "| threads", torch.get_num_threads())

FULL {'valid': 393399, 'test': 378788} | items 101 | threads 10


In [2]:
from nbp.data.query_features import build_vocab, encode
from nbp.hybrid.fusion import cluster_se
from nbp.models import scorers as S
from nbp.models.train import TrainConfig, fit, run_dir_for

train_rows = np.flatnonzero(rows["window"] == TRAIN)
vocab = build_vocab(codes, rows["event"][train_rows])  # train window only
Q = torch.from_numpy(encode(codes, vocab))
field_sizes = [len(v) + 1 for v in vocab]
drop = TRAIN_CFG["dropout_grid"][0]
cfg = S.model_config(MODEL_CFG, {"seed": REF_SEED, "hidden_dropout_prob": drop})
model = S.build_query(cfg, N_ITEMS, Q, field_sizes)
loss = S.query_loss(rows["seq"][train_rows], rows["event"][train_rows], rows["y"][train_rows])
n_ep = int(TRAIN_CFG["calibration_epochs"])
tc = TrainConfig(max_epochs=n_ep, patience=n_ep + 1, min_delta=0.0, nll_guard_epochs=n_ep + 1,
                 batch_size=TRAIN_CFG["batch_size"], learning_rate=TRAIN_CFG["learning_rate"],
                 weight_decay=TRAIN_CFG["weight_decay"], seed=REF_SEED)
ev = R.valid_eval_fn(S.query_scorer, views["valid"], Y["valid"], probe, y_probe, K_MON, BATCH)
per_epoch_ranks = []


def ev_keep(m):
    out = ev(m)
    per_epoch_ranks.append(ev.last_ranks.copy())
    return out


run_dir = run_dir_for(EXP, "expedia", "query-embedding", f"calib-d{drop}-s{REF_SEED}", TODAY)
res = fit(model, len(train_rows), loss, ev_keep, tc, run_dir, resume=False)

epoch 1: train loss 3.0649 (523.6s) | valid monitor 0.4319 nll 3.0217 (35.3s) | best 1, no gain 0/4


epoch 2: train loss 2.7712 (457.8s) | valid monitor 0.4399 nll 2.9431 (51.7s) | best 2, no gain 0/4


epoch 3: train loss 2.7150 (438.6s) | valid monitor 0.4407 nll 2.9515 (42.3s) | best 3, no gain 0/4
WARNING best epoch = max_epochs = 3: possibly under-trained
stop: max_epochs; best epoch 3 (monitor 0.4407)


In [3]:
def per_row_ndcg(r, k):
    return (r <= k) / np.log2(r + 1)


d = per_row_ndcg(per_epoch_ranks[-1], K_MON) - per_row_ndcg(per_epoch_ranks[-2], K_MON)
se = cluster_se(d, USERS["valid"])
min_delta = float(TRAIN_CFG["min_delta_se_multiple"]) * se
lo, hi = TRAIN_CFG["min_delta_bounds"]
calib = {
    "model": "query embedding", "seed": REF_SEED, "hidden_dropout_prob": drop, "epochs": n_ep,
    "monitor": f"autoregressive valid ndcg@{K_MON}",
    "per_epoch": res.per_epoch,
    "gain_last_epoch_mean": float(d.mean()), "gain_se_user_clustered": se,
    "se_multiple": TRAIN_CFG["min_delta_se_multiple"], "min_delta": min_delta, "bounds": [lo, hi],
    "within_bounds": bool(lo <= min_delta <= hi), "rows": int(len(d)), "users": int(len(np.unique(USERS["valid"]))),
}
write_json(RES / "min_delta_calibration.json", calib)
print(json.dumps({k: v for k, v in calib.items() if k != "per_epoch"}, indent=1))
if not calib["within_bounds"] and not FAST:
    raise SystemExit(f"min_delta {min_delta:.2e} outside {lo}-{hi}: stop and report")
if not FAST:
    p = CONFIGS / "train.yaml"
    txt = p.read_text(encoding="utf-8")
    old = [l for l in txt.splitlines() if l.startswith("min_delta:")][0]
    p.write_text(txt.replace(old, f"min_delta: {min_delta:.6g}      # set by notebook 00 ({TODAY}); results/min_delta_calibration.json"), encoding="utf-8")
    print("configs/train.yaml min_delta =", f"{min_delta:.6g}")

{
 "model": "query embedding",
 "seed": 2022,
 "hidden_dropout_prob": 0.0,
 "epochs": 3,
 "monitor": "autoregressive valid ndcg@10",
 "gain_last_epoch_mean": 0.0008353734951158059,
 "gain_se_user_clustered": 0.0003497428558108514,
 "se_multiple": 2.0,
 "min_delta": 0.0006994857116217028,
 "bounds": [
  0.0001,
  0.005
 ],
 "within_bounds": true,
 "rows": 393399,
 "users": 226064
}
configs/train.yaml min_delta = 0.000699486
